In [1]:
import os
import math
import time
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

SEEDS = [42, 123, 999]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FIGURES = '/content/drive/MyDrive/MTech/DNN Assignment 2/problem1/figures'


def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

results_file = '/content/drive/MyDrive/MTech/DNN Assignment 2/problem1/results.csv'
if not os.path.exists(results_file):
    df_init = pd.DataFrame(columns=[
        'problem', 'model', 'seed', 'seq_len', 'test_acc',
        'macro_f1', 'test_mse', 'params', 'sec_per_epoch',
        'val_loss_nats', 'bits_per_char', 'notes'
    ])
    df_init.to_csv(results_file, index=False)

def log_result(problem, model, seed, seq_len=None, test_acc=None, macro_f1=None,
               test_mse=None, params=None, sec_per_epoch=None, notes=None):
    df = pd.read_csv(results_file)
    new_row = {
        'problem': problem, 'model': model, 'seed': seed, 'seq_len': seq_len,
        'test_acc': test_acc, 'macro_f1': macro_f1, 'test_mse': test_mse,
        'params': params, 'sec_per_epoch': sec_per_epoch,
        'val_loss_nats': None, 'bits_per_char': None, 'notes': notes
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(results_file, index=False)

os.makedirs(FIGURES, exist_ok=True)

# 1. TASK 1: DATA PIPELINES


In [2]:
def make_adding(n, T, seed):
    rng = np.random.default_rng(seed)
    x = np.zeros((n, T, 2), dtype=np.float32)
    x[:, :, 0] = rng.random((n, T))
    i1 = rng.integers(0, T // 2, n)
    i2 = rng.integers(T // 2, T, n)
    x[np.arange(n), i1, 1] = 1
    x[np.arange(n), i2, 1] = 1
    y = x[np.arange(n), i1, 0] + x[np.arange(n), i2, 0]
    return x, y.astype(np.float32)

# Unit test for adding problem dataset generator
x_test, y_test = make_adding(1000, 50, seed=42)
assert x_test.shape == (1000, 50, 2)
assert y_test.shape == (1000,)
for idx in range(1000):
    markers = np.where(x_test[idx, :, 1] == 1)[0]
    assert len(markers) == 2, f"Sequence {idx} does not have exactly 2 markers"
    assert markers[0] < 25, f"First marker {markers[0]} not in first half"
    assert markers[1] >= 25, f"Second marker {markers[1]} not in second half"
    expected_sum = x_test[idx, markers[0], 0] + x_test[idx, markers[1], 0]
    assert np.isclose(y_test[idx], expected_sum), f"Target sum mismatch at {idx}"
print("[PASS] Task 1: Adding Problem pipeline test passed.")

def load_har_data(data_dir='har/UCI HAR Dataset'):
    if not os.path.exists(data_dir):
        url = 'https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip'
        urllib.request.urlretrieve(url, 'har.zip')
        with zipfile.ZipFile('har.zip', 'r') as zip_ref:
            zip_ref.extractall('har')
        if os.path.exists('har/UCI HAR Dataset.zip'):
            with zipfile.ZipFile('har/UCI HAR Dataset.zip', 'r') as zip_ref:
                zip_ref.extractall('har')

    signal_names = [
        'body_acc_x_', 'body_acc_y_', 'body_acc_z_',
        'body_gyro_x_', 'body_gyro_y_', 'body_gyro_z_',
        'total_acc_x_', 'total_acc_y_', 'total_acc_z_'
    ]

    def load_signals(split):
        signals = []
        for name in signal_names:
            path = os.path.join(data_dir, split, 'Inertial Signals', f'{name}{split}.txt')
            signals.append(np.loadtxt(path))
        return np.stack(signals, axis=-1)

    X_train = load_signals('train')
    y_train = np.loadtxt(os.path.join(data_dir, 'train', 'y_train.txt')).astype(int) - 1
    sub_train = np.loadtxt(os.path.join(data_dir, 'train', 'subject_train.txt')).astype(int)

    X_test = load_signals('test')
    y_test = np.loadtxt(os.path.join(data_dir, 'test', 'y_test.txt')).astype(int) - 1
    sub_test = np.loadtxt(os.path.join(data_dir, 'test', 'subject_test.txt')).astype(int)

    return X_train, y_train, sub_train, X_test, y_test, sub_test

def get_har_splits(seed=42):
    X_train_raw, y_train_raw, sub_train_raw, X_test_raw, y_test_raw, sub_test_raw = load_har_data()
    unique_subs = np.unique(sub_train_raw)
    rng = np.random.default_rng(seed)
    val_subs = rng.choice(unique_subs, 4, replace=False)

    val_mask = np.isin(sub_train_raw, val_subs)
    train_mask = ~val_mask

    X_tr, y_tr = X_train_raw[train_mask], y_train_raw[train_mask]
    X_va, y_va = X_train_raw[val_mask], y_train_raw[val_mask]
    X_te, y_te = X_test_raw, y_test_raw

    # Per-channel mean & std computed strictly from training split
    mean = X_tr.mean(axis=(0, 1), keepdims=True)
    std = X_tr.std(axis=(0, 1), keepdims=True) + 1e-8

    X_tr = (X_tr - mean) / std
    X_va = (X_va - mean) / std
    X_te = (X_te - mean) / std

    return (X_tr, y_tr), (X_va, y_va), (X_te, y_te)

# Plot class balance for HAR dataset
try:
    X_tr, y_tr = load_har_data()[0], load_har_data()[1]
    plt.figure(figsize=(7, 4))
    classes, counts = np.unique(y_tr, return_counts=True)
    plt.bar(['Walking', 'Upstairs', 'Downstairs', 'Sitting', 'Standing', 'Laying'], counts, color='skyblue', edgecolor='black')
    plt.title('UCI HAR Training Class Distribution')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig(FIGURES+'/har_class_balance.png')
    plt.close()
    print("[INFO] HAR Class balance plot saved to figures/har_class_balance.png.")
except Exception as e:
    print(f"[WARN] Skipping HAR plot: {e}")

[PASS] Task 1: Adding Problem pipeline test passed.
[INFO] HAR Class balance plot saved to figures/har_class_balance.png.


# 2. TASK 2: CELLS FROM SCRATCH


In [3]:
class MyRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.W_ih = nn.Parameter(torch.randn(hidden_size, input_size) * 0.01)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        self.b_ih = nn.Parameter(torch.zeros(hidden_size))
        self.b_hh = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x, h):
        return torch.tanh(x @ self.W_ih.T + self.b_ih + h @ self.W_hh.T + self.b_hh)

class MyLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size, forget_gate_bias=0.0):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.W_ih = nn.Parameter(torch.randn(4 * hidden_size, input_size) * 0.01)
        self.W_hh = nn.Parameter(torch.randn(4 * hidden_size, hidden_size) * 0.01)
        self.b_ih = nn.Parameter(torch.zeros(4 * hidden_size))
        self.b_hh = nn.Parameter(torch.zeros(4 * hidden_size))
        if forget_gate_bias != 0.0:
            with torch.no_grad():
                self.b_ih[hidden_size:2*hidden_size].fill_(forget_gate_bias)

    def forward(self, x, states):
        h, c = states
        gates = x @ self.W_ih.T + self.b_ih + h @ self.W_hh.T + self.b_hh
        i, f, g, o = torch.chunk(gates, 4, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

class MyGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.W_ih = nn.Parameter(torch.randn(3 * hidden_size, input_size) * 0.01)
        self.W_hh = nn.Parameter(torch.randn(3 * hidden_size, hidden_size) * 0.01)
        self.b_ih = nn.Parameter(torch.zeros(3 * hidden_size))
        self.b_hh = nn.Parameter(torch.zeros(3 * hidden_size))

    def forward(self, x, h):
        gi = x @ self.W_ih.T + self.b_ih
        gh = h @ self.W_hh.T + self.b_hh
        r_i, z_i, n_i = torch.chunk(gi, 3, dim=1)
        r_h, z_h, n_h = torch.chunk(gh, 3, dim=1)
        r = torch.sigmoid(r_i + r_h)
        z = torch.sigmoid(z_i + z_h)
        n = torch.tanh(n_i + r * n_h)
        return (1 - z) * n + z * h

# Unit tests to verify custom cells against official PyTorch implementations (< 1e-5 difference)
seed_everything(42)
d, h = 9, 64
x_test = torch.randn(16, d)
h_test = torch.randn(16, h)
c_test = torch.randn(16, h)

# --- Test MyRNNCell ---
my_rnn = MyRNNCell(d, h)
py_rnn = nn.RNNCell(d, h)
with torch.no_grad():
    py_rnn.weight_ih.copy_(my_rnn.W_ih)
    py_rnn.weight_hh.copy_(my_rnn.W_hh)
    py_rnn.bias_ih.copy_(my_rnn.b_ih)
    py_rnn.bias_hh.copy_(my_rnn.b_hh)
    out_my_rnn = my_rnn(x_test, h_test)
    out_py_rnn = py_rnn(x_test, h_test)
    assert torch.allclose(out_my_rnn, out_py_rnn, atol=1e-5), "MyRNNCell verification failed!"

# --- Test MyLSTMCell ---
my_lstm = MyLSTMCell(d, h)
py_lstm = nn.LSTMCell(d, h)
with torch.no_grad():
    py_lstm.weight_ih.copy_(my_lstm.W_ih)
    py_lstm.weight_hh.copy_(my_lstm.W_hh)
    py_lstm.bias_ih.copy_(my_lstm.b_ih)
    py_lstm.bias_hh.copy_(my_lstm.b_hh)
    h_my, c_my = my_lstm(x_test, (h_test, c_test))
    h_py, c_py = py_lstm(x_test, (h_test, c_test))
    assert torch.allclose(h_my, h_py, atol=1e-5) and torch.allclose(c_my, c_py, atol=1e-5), "MyLSTMCell verification failed!"

# --- Test MyGRUCell ---
my_gru = MyGRUCell(d, h)
py_gru = nn.GRUCell(d, h)
with torch.no_grad():
    py_gru.weight_ih.copy_(my_gru.W_ih)
    py_gru.weight_hh.copy_(my_gru.W_hh)
    py_gru.bias_ih.copy_(my_gru.b_ih)
    py_gru.bias_hh.copy_(my_gru.b_hh)
    out_my_gru = my_gru(x_test, h_test)
    out_py_gru = py_gru(x_test, h_test)
    assert torch.allclose(out_my_gru, out_py_gru, atol=1e-5), "MyGRUCell verification failed!"

print("[PASS] Task 2: All custom cells verified against PyTorch reference within 1e-5 tolerance.")

[PASS] Task 2: All custom cells verified against PyTorch reference within 1e-5 tolerance.


# 3. SEQUENCE CLASSIFIER AND REGRESSOR MODELS


In [4]:
class HARClassifier(nn.Module):
    def __init__(self, cell_type, input_size=9, hidden_size=64, num_classes=6, forget_bias=0.0):
        super().__init__()
        self.cell_type = cell_type
        self.hidden_size = hidden_size
        if cell_type == 'RNN':
            self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        elif cell_type == 'LSTM':
            self.rnn = nn.LSTM(input_size, hidden_size, batch_first=True)
            if forget_bias != 0.0:
                with torch.no_grad():
                    self.rnn.bias_ih_l0[hidden_size:2*hidden_size].fill_(forget_bias)
        elif cell_type == 'GRU':
            self.rnn = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.rnn(x)
        last_h = out[:, -1, :]
        return self.fc(last_h)

class AddingRegressor(nn.Module):
    def __init__(self, cell_type, input_size=2, hidden_size=64, forget_bias=0.0):
        super().__init__()
        self.cell_type = cell_type
        self.hidden_size = hidden_size
        if cell_type == 'RNN':
            self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        elif cell_type == 'LSTM':
            self.rnn = nn.LSTM(input_size, hidden_size, batch_first=True)
            if forget_bias != 0.0:
                with torch.no_grad():
                    self.rnn.bias_ih_l0[hidden_size:2*hidden_size].fill_(forget_bias)
        elif cell_type == 'GRU':
            self.rnn = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        last_h = out[:, -1, :]
        return self.fc(last_h).squeeze(-1)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# 4. TASK 3: HAR CLASSIFICATION


In [5]:
def train_har(model_name, hidden_size=64, epochs=30, batch_size=64):
    from sklearn.metrics import f1_score, confusion_matrix
    print(f"\n--- Running HAR Classification: {model_name} (Hidden Size: {hidden_size}) ---")

    test_accs, macro_f1s = [], []
    best_cm = None
    best_acc = -1.0

    for seed in SEEDS:
        seed_everything(seed)
        (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = get_har_splits(seed=seed)

        train_loader = DataLoader(TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long)), batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(TensorDataset(torch.tensor(X_te, dtype=torch.float32), torch.tensor(y_te, dtype=torch.long)), batch_size=batch_size, shuffle=False)

        model = HARClassifier(model_name, input_size=9, hidden_size=hidden_size, num_classes=6).to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        t0 = time.time()
        for epoch in range(epochs):
            model.train()
            for bx, by in train_loader:
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(bx), by)
                loss.backward()
                optimizer.step()
        sec_per_epoch = (time.time() - t0) / epochs

        # Test evaluation
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for bx, by in test_loader:
                bx = bx.to(DEVICE)
                out = model(bx)
                preds.append(out.argmax(dim=1).cpu().numpy())
                targets.append(by.numpy())

        preds = np.concatenate(preds)
        targets = np.concatenate(targets)

        acc = (preds == targets).mean() * 100
        f1 = f1_score(targets, preds, average='macro')

        test_accs.append(acc)
        macro_f1s.append(f1)

        params = count_parameters(model)
        log_result("P1_Task3", model_name, seed, seq_len=128, test_acc=acc, macro_f1=f1, params=params, sec_per_epoch=sec_per_epoch)

        if acc > best_acc:
            best_acc = acc
            best_cm = confusion_matrix(targets, preds)

    print(f"{model_name} HAR Results -> Acc: {np.mean(test_accs):.2f}% +/- {np.std(test_accs):.2f}%, F1: {np.mean(macro_f1s):.4f} +/- {np.std(macro_f1s):.4f}")
    return best_cm

cm_gru = None
for m in ['RNN', 'LSTM', 'GRU']:
    cm = train_har(m, hidden_size=64, epochs=25)
    if m == 'GRU':
        cm_gru = cm

# Plot best confusion matrix
if cm_gru is not None:
    plt.figure(figsize=(6, 5))
    plt.imshow(cm_gru, cmap='Blues')
    plt.title('Best HAR Model Confusion Matrix (GRU)')
    plt.colorbar()
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig(FIGURES+'/har_confusion_matrix.png')
    plt.close()


--- Running HAR Classification: RNN (Hidden Size: 64) ---


/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA 

RNN HAR Results -> Acc: 77.57% +/- 3.84%, F1: 0.7708 +/- 0.0414

--- Running HAR Classification: LSTM (Hidden Size: 64) ---


/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA 

LSTM HAR Results -> Acc: 89.06% +/- 0.72%, F1: 0.8909 +/- 0.0074

--- Running HAR Classification: GRU (Hidden Size: 64) ---


/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA 

GRU HAR Results -> Acc: 90.77% +/- 0.68%, F1: 0.9085 +/- 0.0071


# 5. TASK 4: ADDING PROBLEM EXPERIMENTS


In [6]:
def train_adding(lengths=[50, 100, 200, 400], epochs=20, batch_size=128):
    print("\n--- Running Adding Problem Experiments ---")
    results_dict = {m: {} for m in ['RNN', 'LSTM', 'GRU']}
    learning_curves_t200 = {m: [] for m in ['RNN', 'LSTM', 'GRU']}

    for T in lengths:
        for model_name in ['RNN', 'LSTM', 'GRU']:
            mses = []
            for seed in SEEDS:
                seed_everything(seed)
                X_tr, y_tr = make_adding(10000, T, seed=0)
                X_te, y_te = make_adding(1000, T, seed=2)

                train_loader = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)), batch_size=batch_size, shuffle=True)
                test_loader = DataLoader(TensorDataset(torch.tensor(X_te), torch.tensor(y_te)), batch_size=batch_size, shuffle=False)

                model = AddingRegressor(model_name, input_size=2, hidden_size=64).to(DEVICE)
                optimizer = optim.Adam(model.parameters(), lr=1e-3)
                criterion = nn.MSELoss()

                curve = []
                t0 = time.time()
                for epoch in range(epochs):
                    model.train()
                    run_loss = 0.0
                    for bx, by in train_loader:
                        bx, by = bx.to(DEVICE), by.to(DEVICE)
                        optimizer.zero_grad()
                        loss = criterion(model(bx), by)
                        loss.backward()
                        optimizer.step()
                        run_loss += loss.item() * len(by)
                    curve.append(run_loss / len(X_tr))
                sec_per_epoch = (time.time() - t0) / epochs

                model.eval()
                test_loss = 0.0
                with torch.no_grad():
                    for bx, by in test_loader:
                        bx, by = bx.to(DEVICE), by.to(DEVICE)
                        test_loss += criterion(model(bx), by).item() * len(by)
                final_mse = test_loss / len(X_te)
                mses.append(final_mse)

                if T == 200:
                    learning_curves_t200[model_name].append(curve)

                log_result("P1_Task4", model_name, seed, seq_len=T, test_mse=final_mse, params=count_parameters(model), sec_per_epoch=sec_per_epoch)

            solved_count = sum(1 for m in mses if m < 0.05)
            results_dict[model_name][T] = (np.mean(mses), np.std(mses), solved_count)
            print(f"T={T:3d} | {model_name:4s} -> MSE: {np.mean(mses):.4f} +/- {np.std(mses):.4f} | Solved: {solved_count}/3")

    # Plot learning curves at T = 200
    plt.figure(figsize=(7, 4))
    for m in ['RNN', 'LSTM', 'GRU']:
        avg_curve = np.mean(learning_curves_t200[m], axis=0)
        plt.plot(avg_curve, label=m)
    plt.axhline(0.167, color='red', linestyle='--', label='Baseline (0.167)')
    plt.xlabel('Epoch')
    plt.ylabel('Training MSE')
    plt.title('Adding Problem Learning Curves (T = 200)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES+'/adding_learning_curves_t200.png')
    plt.close()

train_adding(lengths=[50, 100, 200, 400], epochs=20)


--- Running Adding Problem Experiments ---


/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA 

T= 50 | RNN  -> MSE: 0.1745 +/- 0.0004 | Solved: 0/3


/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA 

T= 50 | LSTM -> MSE: 0.0702 +/- 0.0731 | Solved: 2/3


/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA 

... (15 outputs left out)

/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA 

T=400 | LSTM -> MSE: 0.1566 +/- 0.0045 | Solved: 0/3


/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_1296/3885614198.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA 

T=400 | GRU  -> MSE: 0.1503 +/- 0.0014 | Solved: 0/3


# 6. TASK 5
## (a): GRADIENT DYNAMICS (T = 100)
## (b): HIDDEN SIZE SWEEP ON HAR
## (c): INDIVIDUAL ABLATION
- (ID Even: Gradient Clipping vs None for RNN at T = 200)
- (ID Odd:  LSTM Forget-gate Bias 0 vs 1 at T = 200)


In [7]:
def analyze_gradients(T=100):
    print("\n--- Running Task 5(a): Gradient Analysis (T = 100) ---")
    seed_everything(42)
    X_sample, y_sample = make_adding(1, T, seed=0)
    bx = torch.tensor(X_sample, dtype=torch.float32, requires_grad=True).to(DEVICE)
    by = torch.tensor(y_sample, dtype=torch.float32).to(DEVICE)

    plt.figure(figsize=(8, 4))
    for model_name in ['RNN', 'LSTM', 'GRU']:
        model = AddingRegressor(model_name, input_size=2, hidden_size=64).to(DEVICE)

        # Unroll manually to get gradient wrt hidden state at each timestep
        out, _ = model.rnn(bx) # out: [1, T, 64]
        loss = (model.fc(out[:, -1, :]).squeeze(-1) - by) ** 2

        grad_norms = []
        for t in range(T):
            h_t = out[:, t, :]
            grad_h = torch.autograd.grad(loss, h_t, retain_graph=True, allow_unused=True)[0]
            if grad_h is None:
                grad_h = torch.zeros_like(h_t)
            grad_norms.append(grad_h.norm().item())

        plt.plot(range(T), grad_norms, label=f'{model_name} (Untrained)')

    plt.yscale('log')
    plt.xlabel('Timestep t')
    plt.ylabel(r'$\|\partial L / \partial h_t\|$')
    plt.title('Gradient Norm Dynamics wrt Timestep (T = 100)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES+'/gradient_norms_t100.png')
    plt.close()

analyze_gradients(T=100)


def hidden_size_sweep(sizes=[16, 32, 64, 128]):
    print("\n--- Running Task 5(b): Hidden Size Sweep ---")
    sweep_results = {'LSTM': [], 'GRU': []}

    for m in ['LSTM', 'GRU']:
        for h in sizes:
            accs = []
            for seed in SEEDS:
                seed_everything(seed)
                (X_tr, y_tr), _, (X_te, y_te) = get_har_splits(seed=seed)
                train_loader = DataLoader(TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long)), batch_size=64, shuffle=True)
                test_loader = DataLoader(TensorDataset(torch.tensor(X_te, dtype=torch.float32), torch.tensor(y_te, dtype=torch.long)), batch_size=64, shuffle=False)

                model = HARClassifier(m, input_size=9, hidden_size=h, num_classes=6).to(DEVICE)
                optimizer = optim.Adam(model.parameters(), lr=1e-3)
                criterion = nn.CrossEntropyLoss()

                for epoch in range(15):
                    model.train()
                    for bx, by in train_loader:
                        bx, by = bx.to(DEVICE), by.to(DEVICE)
                        optimizer.zero_grad()
                        loss = criterion(model(bx), by)
                        loss.backward()
                        optimizer.step()

                model.eval()
                correct = 0
                with torch.no_grad():
                    for bx, by in test_loader:
                        bx, by = bx.to(DEVICE), by.to(DEVICE)
                        correct += (model(bx).argmax(dim=1) == by).sum().item()
                accs.append(correct / len(y_te) * 100)

            mean_acc = np.mean(accs)
            sweep_results[m].append(mean_acc)
            print(f"{m} (Hidden Size {h:3d}) -> Test Acc: {mean_acc:.2f}%")

    plt.figure(figsize=(7, 4))
    for m in ['LSTM', 'GRU']:
        plt.plot(sizes, sweep_results[m], marker='o', label=m)
    plt.xlabel('Hidden Size h')
    plt.ylabel('Test Accuracy (%)')
    plt.title('HAR Accuracy vs Hidden Size Sweep')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES+'/hidden_size_sweep.png')
    plt.close()

hidden_size_sweep([16, 32, 64, 128])


def run_ablation(is_even_student_id=True, T=200, epochs=20):
    print(f"\n--- Running Task 5(c): Ablation Study (T={T}) ---")
    plt.figure(figsize=(7, 4))

    if is_even_student_id:
        print("Ablation: Gradient Clipping (norm 1.0) vs None for Vanilla RNN")
        for clip_val in [None, 1.0]:
            mses_per_epoch = []
            for seed in SEEDS:
                seed_everything(seed)
                X_tr, y_tr = make_adding(10000, T, seed=0)
                train_loader = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)), batch_size=128, shuffle=True)

                model = AddingRegressor('RNN', input_size=2, hidden_size=64).to(DEVICE)
                optimizer = optim.Adam(model.parameters(), lr=1e-3)
                criterion = nn.MSELoss()

                epoch_mses = []
                for epoch in range(epochs):
                    model.train()
                    run_loss = 0.0
                    for bx, by in train_loader:
                        bx, by = bx.to(DEVICE), by.to(DEVICE)
                        optimizer.zero_grad()
                        loss = criterion(model(bx), by)
                        loss.backward()
                        if clip_val is not None:
                            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_val)
                        optimizer.step()
                        run_loss += loss.item() * len(by)
                    epoch_mses.append(run_loss / len(X_tr))
                mses_per_epoch.append(epoch_mses)

            label = "Grad Clip 1.0" if clip_val == 1.0 else "No Clipping"
            plt.plot(np.mean(mses_per_epoch, axis=0), label=label)
    else:
        print("Ablation: LSTM Forget-gate Bias Initialized to 0 vs 1")
        for f_bias in [0.0, 1.0]:
            mses_per_epoch = []
            for seed in SEEDS:
                seed_everything(seed)
                X_tr, y_tr = make_adding(10000, T, seed=0)
                train_loader = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)), batch_size=128, shuffle=True)

                model = AddingRegressor('LSTM', input_size=2, hidden_size=64, forget_bias=f_bias).to(DEVICE)
                optimizer = optim.Adam(model.parameters(), lr=1e-3)
                criterion = nn.MSELoss()

                epoch_mses = []
                for epoch in range(epochs):
                    model.train()
                    run_loss = 0.0
                    for bx, by in train_loader:
                        bx, by = bx.to(DEVICE), by.to(DEVICE)
                        optimizer.zero_grad()
                        loss = criterion(model(bx), by)
                        loss.backward()
                        optimizer.step()
                        run_loss += loss.item() * len(by)
                    epoch_mses.append(run_loss / len(X_tr))
                mses_per_epoch.append(epoch_mses)

            plt.plot(np.mean(mses_per_epoch, axis=0), label=f"Forget Bias = {f_bias}")

    plt.axhline(0.167, color='red', linestyle='--', label='Baseline')
    plt.xlabel('Epoch')
    plt.ylabel('Training MSE')
    plt.title('Ablation Study Learning Curves (T = 200)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES+'/ablation_study.png')
    plt.close()
    print("[INFO] Ablation plot saved to figures/ablation_study.png.")

# Run ablation (Default set to Even ID, change to False for Odd ID)
run_ablation(is_even_student_id=True, T=200)

print("\n==========================================")
print("Problem 1 Execution Completed Successfully!")
print("All metrics logged to results.csv")
print("==========================================")


--- Running Task 5(a): Gradient Analysis (T = 100) ---


/tmp/ipykernel_1296/1956266155.py:26: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  plt.yscale('log')



--- Running Task 5(b): Hidden Size Sweep ---
LSTM (Hidden Size  16) -> Test Acc: 79.83%
LSTM (Hidden Size  32) -> Test Acc: 85.48%
LSTM (Hidden Size  64) -> Test Acc: 87.85%
LSTM (Hidden Size 128) -> Test Acc: 87.59%
GRU (Hidden Size  16) -> Test Acc: 86.33%
GRU (Hidden Size  32) -> Test Acc: 89.23%
GRU (Hidden Size  64) -> Test Acc: 90.77%
GRU (Hidden Size 128) -> Test Acc: 90.34%

--- Running Task 5(c): Ablation Study (T=200) ---
Ablation: Gradient Clipping (norm 1.0) vs None for Vanilla RNN
[INFO] Ablation plot saved to figures/ablation_study.png.

Problem 1 Execution Completed Successfully!
All metrics logged to results.csv
